# Paso por Referencia en C++

**Curso:** Programación Avanzada  
**Sesión:** 17  
**Kernel:** xeus-cling (C++17)

---

En la sesión anterior estudiamos apuntadores, referencias y arreglos.  
Hoy aplicamos esos conceptos al **paso de parámetros en funciones**, que es donde cobran todo su sentido práctico.

Al terminar la sesión veremos el algoritmo **Bubble Sort** y quedarán pendientes como práctica las funciones `swap`, `comparar` y `bubblesort`.

In [ ]:
#include <iostream>
#include <string>
using namespace std;

---

## Parte 1 — El problema con el paso por valor

Por defecto, C++ pasa los argumentos **por valor**: la función recibe una *copia* del dato original. Cualquier modificación dentro de la función no afecta al original.

In [ ]:
{
    void intentaCambiar(int n) {
        n = 999;   // solo modifica la copia local
    }

    int x = 42;
    intentaCambiar(x);
    cout << "x después de intentaCambiar: " << x << endl;  // sigue siendo 42
}

**¿Por qué no cambia?**  
Cuando se llama `intentaCambiar(x)`, el compilador crea una nueva variable `n` en el stack de la función y copia el valor de `x` en ella. `n` y `x` son celdas de memoria distintas. Modificar `n` no toca `x`.

```
  main()           intentaCambiar()
  ┌────────┐       ┌────────┐
  │  x=42  │  →   │  n=42  │   (copia)
  └────────┘       └────────┘
  0xAA00           0xBB00        (direcciones distintas)
```

---

## Parte 2 — Paso por referencia

Cuando declaramos el parámetro con `&`, la función recibe un **alias** de la variable original, no una copia. Ambos nombres apuntan a la misma celda de memoria.

In [ ]:
{
    void cambiaConReferencia(int& n) {   // n es alias del argumento
        n = 999;
    }

    int x = 42;
    cout << "x antes:  " << x << endl;
    cambiaConReferencia(x);
    cout << "x después: " << x << endl;  // ahora sí cambió
}

**¿Por qué ahora sí cambia?**  
Con `int& n`, el compilador no crea una copia: `n` es simplemente otro nombre para la misma celda de memoria donde vive `x`. Escribir `n = 999` es exactamente lo mismo que escribir `x = 999`.

```
  main()           cambiaConReferencia()
  ┌────────┐
  │  x=42  │ ←──── n   (mismo espacio, dos nombres)
  └────────┘
  0xAA00
```

En la llamada `cambiaConReferencia(x)` no se escribe `&x`: el `&` va solo en la **declaración** del parámetro, no en la llamada.

---

## Parte 3 — Paso por apuntador

Antes de que C++ introdujera las referencias (C las heredó de C), la única forma de lograr el mismo efecto era pasar la **dirección** de la variable y trabajar con un apuntador dentro de la función.

In [ ]:
{
    void cambiaConApuntador(int* p) {   // p recibe una dirección
        *p = 999;                        // desreferenciamos para modificar el original
    }

    int x = 42;
    cout << "x antes:  " << x << endl;
    cambiaConApuntador(&x);              // pasamos la dirección de x
    cout << "x después: " << x << endl;
}

El resultado es el mismo, pero la sintaxis es más ruidosa: hay que escribir `&x` en la llamada y `*p` dentro de la función. Las referencias hacen lo mismo con sintaxis más limpia.

### Comparación lado a lado

In [ ]:
{
    // Por valor       — recibe copia, no modifica el original
    void porValor(int n)  { n = 0; }

    // Por referencia  — recibe alias, sí modifica el original
    void porRef(int& n)   { n = 0; }

    // Por apuntador   — recibe dirección, sí modifica el original
    void porPtr(int* p)   { *p = 0; }

    int a = 10, b = 10, c = 10;

    porValor(a);
    porRef(b);
    porPtr(&c);

    cout << "porValor: a = " << a << endl;  // 10
    cout << "porRef:   b = " << b << endl;  // 0
    cout << "porPtr:   c = " << c << endl;  // 0
}

---

## Parte 4 — Cuándo usar cada mecanismo

| Mecanismo | Cuándo usarlo |
|---|---|
| Por valor | El dato es pequeño (int, double, char) y la función no necesita modificarlo |
| Por referencia (`&`) | La función necesita modificar el original, o el dato es grande y no queremos copiarlo |
| Por referencia constante (`const&`) | El dato es grande y la función solo lo lee (evita copias sin riesgo de modificar) |
| Por apuntador (`*`) | Cuando puede ser `nullptr`, o cuando se trabaja con arreglos / memoria dinámica |

### Referencia constante — leer sin copiar

In [ ]:
{
    // Imprimir un string sin copiarlo
    void imprimir(const string& texto) {
        // texto = "otro";  // error de compilación: es const
        cout << texto << endl;
    }

    string saludo = "Hola, mundo";
    imprimir(saludo);
}

**¿Por qué es eficiente?**  
Un `string` puede ocupar mucho espacio en memoria. Pasarlo por valor crearía una copia completa. Con `const string&` pasamos solo la dirección (8 bytes) y el `const` garantiza que la función no puede modificar el original.

---

## Parte 5 — Funciones que devuelven múltiples valores

Una función solo puede tener un `return`. El paso por referencia permite "devolver" varios resultados usando parámetros de salida.

In [ ]:
{
    // Calcula el mínimo y máximo de un arreglo a la vez
    void minMax(const int* arr, int n, int& minVal, int& maxVal) {
        minVal = maxVal = arr[0];
        for (int i = 1; i < n; i++) {
            if (arr[i] < minVal) minVal = arr[i];
            if (arr[i] > maxVal) maxVal = arr[i];
        }
    }

    int datos[6] = {3, 7, 1, 9, 4, 6};
    int minimo, maximo;

    minMax(datos, 6, minimo, maximo);

    cout << "Mínimo: " << minimo << endl;
    cout << "Máximo: " << maximo << endl;
}

`minVal` y `maxVal` son referencias a las variables `minimo` y `maximo` del llamador. La función escribe en ellas directamente, logrando el efecto de devolver dos valores.

---

## Parte 6 — Paso de arreglos a funciones

Los arreglos **siempre** se pasan por referencia implícita: el nombre del arreglo decae a un apuntador al primer elemento, por lo que la función trabaja sobre el arreglo original.

In [ ]:
{
    void duplicar(int* arr, int n) {
        for (int i = 0; i < n; i++)
            arr[i] *= 2;   // modifica el arreglo original
    }

    void imprimir(const int* arr, int n) {
        for (int i = 0; i < n; i++)
            cout << arr[i] << " ";
        cout << endl;
    }

    int nums[5] = {1, 2, 3, 4, 5};

    cout << "Antes:   "; imprimir(nums, 5);
    duplicar(nums, 5);
    cout << "Después: "; imprimir(nums, 5);
}

**¿Por qué hay que pasar también el tamaño `n`?**  
Porque al llegar a la función, el arreglo ya decayó a un apuntador y se perdió la información del tamaño (como vimos en la sesión anterior con `sizeof`). La función no tiene forma de saber cuántos elementos tiene si no se lo indicamos.

---

## Parte 7 — Bubble Sort

### Idea del algoritmo

Bubble Sort ordena un arreglo comparando pares de elementos adyacentes e intercambiándolos si están en el orden equivocado. Repite este proceso hasta que no haya más intercambios necesarios.

```
Arreglo inicial:  [5, 3, 8, 1, 4]

Pasada 1:
  [5, 3, ...]  → 5 > 3, intercambia  → [3, 5, 8, 1, 4]
  [3, 5, 8, ...]  → 5 < 8, no toca  → [3, 5, 8, 1, 4]
  [3, 5, 8, 1, ...]  → 8 > 1, intercambia  → [3, 5, 1, 8, 4]
  [3, 5, 1, 8, 4]  → 8 > 4, intercambia  → [3, 5, 1, 4, 8]  ← el 8 llegó al final

Pasada 2:
  → [3, 1, 4, 5, 8]  ← el 5 llegó a su lugar

...y así hasta que ningún par necesite intercambio.
```

El nombre viene de que los elementos grandes "burbujean" hacia el final en cada pasada.

### Las tres piezas que necesitamos

Para implementar Bubble Sort de forma limpia vamos a separarlo en tres funciones:

```
bubblesort(arr, n)
│
├── comparar(a, b)   → bool: ¿a debe ir antes que b?
│
└── swap(a, b)       → intercambia dos elementos usando paso por referencia
```

La función `swap` es el ejercicio perfecto de paso por referencia: necesita **modificar** los dos argumentos originales.

---

## Práctica

Implementa las tres funciones y el programa de prueba. Usa los conceptos de las sesiones 16 y 17:
- `swap` debe usar paso por referencia
- `bubblesort` debe recibir el arreglo como apuntador
- Puedes agregar una función `imprimir` auxiliar

### Función `swap`
Intercambia los valores de dos variables enteras.  
*Pista: necesita una variable auxiliar temporal.*

In [ ]:
// Implementa swap
// void swap(int& a, int& b) { ... }


### Función `comparar`
Decide si el primer elemento debe ir antes que el segundo en el orden deseado.  
Devuelve `true` si están en el orden incorrecto (hay que intercambiar).

In [ ]:
// Implementa comparar
// bool comparar(int a, int b) { ... }


### Función `bubblesort`
Ordena el arreglo usando las funciones `comparar` y `swap`.  
*Pista: necesitas dos ciclos anidados. El externo controla las pasadas; el interno recorre los pares adyacentes.*  
*Pista2: también puedes hacerlo por referencia.*

In [ ]:
// Implementa bubblesort
// void bubblesort(int* arr, int n) { ... }


### Prueba
Una vez implementadas las tres funciones, pruébalas con este arreglo:

In [ ]:
{
    int arr[8] = {64, 25, 12, 22, 11, 90, 3, 47};
    int n = 8;

    // imprimir antes
    cout << "Antes:   ";
    for (int i = 0; i < n; i++) cout << arr[i] << " ";
    cout << endl;

    bubblesort(arr, n);

    // imprimir después
    cout << "Después: ";
    for (int i = 0; i < n; i++) cout << arr[i] << " ";
    cout << endl;
}